# Витрина транзакций ТЭ — 4. Витрина

AI-14134. Одна строка = одна операция. Таблица `sandbox_ai.tmp_shestopalov_acq_mart`.

Факт + клиент по `inn` + терминал по `id_pos_terminal`. Сначала должны быть собраны `01`, `02` и `03`.

Перед сборкой проверяется, что факт не пуст и в клиенте и терминале нет дублей ключа. После сборки — что строк в витрине столько же, сколько в факте.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
with imp:
    keys = imp.fetch("""
select f.fact_rows, c.client_dup, t.terminal_dup
from (select count(*) as fact_rows from sandbox_ai.tmp_shestopalov_acq_trx) f
cross join (
  select count(*) - count(distinct inn) as client_dup
  from sandbox_ai.tmp_shestopalov_acq_clients
) c
cross join (
  select count(*) - count(distinct id_pos_terminal) as terminal_dup
  from sandbox_ai.tmp_shestopalov_acq_terminals
) t
""")
display(keys)
assert keys.loc[0, "fact_rows"] > 0, "Факт пуст: сначала выполните 01_te_trx.ipynb"
assert keys.loc[0, "client_dup"] == 0, "В клиенте дубли inn"
assert keys.loc[0, "terminal_dup"] == 0, "В терминале дубли id_pos_terminal"


In [ ]:
with imp:
    imp.execute("set MEM_LIMIT=8g")
    imp.execute("drop table if exists sandbox_ai.tmp_shestopalov_acq_mart")
    imp.execute("""
create table sandbox_ai.tmp_shestopalov_acq_mart
as
select
  f.inn,
  f.id_pos_terminal,
  f.mcc_code,
  f.mcc_terminal,
  f.mcc_profile,
  f.mcc_mismatch,
  f.transaction_id,
  f.date_t,
  f.sum_t,
  f.status,
  f.card_bin,
  f.type_t,
  f.payment_system,
  f.n_agr,
  f.retl_id,
  f.mrc_city,
  c.okved,
  c.okopf,
  c.region,
  c.filial,
  c.ssp_ocrm,
  c.business_size,
  c.business_size_is_proxy,
  c.ogrn,
  c.jur_address,
  c.date_acquiring,
  c.date_agr_current,
  tr.version,
  tr.date_on,
  tr.date_off
from sandbox_ai.tmp_shestopalov_acq_trx f
left join sandbox_ai.tmp_shestopalov_acq_clients c
  on c.inn = f.inn
left join sandbox_ai.tmp_shestopalov_acq_terminals tr
  on tr.id_pos_terminal = f.id_pos_terminal
""")
print("собрано sandbox_ai.tmp_shestopalov_acq_mart")


In [ ]:
with imp:
    rows = imp.fetch("""
select m.mart_rows, f.fact_rows
from (select count(*) as mart_rows from sandbox_ai.tmp_shestopalov_acq_mart) m
cross join (select count(*) as fact_rows from sandbox_ai.tmp_shestopalov_acq_trx) f
""")
    missing = imp.fetch("""
select
  sum(case when date_acquiring is null then 1 else 0 end) as no_client,
  sum(case when tr.id_pos_terminal is null then 1 else 0 end) as no_terminal
from sandbox_ai.tmp_shestopalov_acq_mart m
left join sandbox_ai.tmp_shestopalov_acq_terminals tr
  on tr.id_pos_terminal = m.id_pos_terminal
""")
    sample = imp.fetch("select * from sandbox_ai.tmp_shestopalov_acq_mart limit 20")
display(rows)
display(missing)
display(sample)
assert rows.loc[0, "mart_rows"] == rows.loc[0, "fact_rows"], "Строк в витрине не столько, сколько в факте"
